<a href="https://colab.research.google.com/github/DymaStar/melbourne-housing/blob/main/notebooks/melbourne_housing_template_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Прогнозування цін на житло в Мельбурні

## Навчальний проєкт Data Analytics / Machine Learning

У цьому ноутбуці виконується повний цикл роботи з даними:

```text
Завантаження даних
        ↓
Перший огляд
        ↓
Очищення
        ↓
EDA — дослідницький аналіз
        ↓
Feature Engineering
        ↓
Підготовка Pipeline
        ↓
Навчання моделей
        ↓
Оцінка якості
        ↓
Прогноз для одного прикладу
        ↓
Висновки
```

🎯 **Мета проєкту:** побудувати модель, яка прогнозує ціну житла в Мельбурні за характеристиками об'єкта.

📌 **Тип задачі:** регресія, тому що прогнозуємо числове значення — `Price`.

🔁 **Аналогія з попередніми роботами:** раніше ми будували Pipeline для класифікації, де прогноз був `0/1`. Тут логіка схожа, але модель прогнозує не клас, а число — ціну житла.

## Крок 0. Імпорт бібліотек

🎯 **Мета:** підключити основні інструменти для роботи з таблицями, графіками та моделями.

- `pandas` — робота з таблицями.
- `numpy` — робота з числами та пропусками.
- `matplotlib` — побудова графіків.
- `sklearn` — моделі машинного навчання, Pipeline, метрики.

💡 **Запам'ятай:** імпорти зазвичай виносять у першу кодову комірку ноутбука.

In [ ]:
# import — це підключення інструментів, які будемо використовувати далі

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Щоб графіки одразу відображались у ноутбуці
%matplotlib inline

# Фіксуємо випадковість, щоб результати щоразу були однакові
RANDOM_STATE = 42

## Крок 1. Завантаження даних

🎯 **Мета:** завантажити файл `melb_data.csv` з GitHub у Google Colab.

Дані потрібно читати через **RAW-посилання**. Звичайне посилання GitHub веде на сторінку з оформленням, а RAW-посилання дає чистий CSV-файл.

⚠️ **Важливо:** заміни `RAW_URL` на своє посилання з GitHub.

Приклад RAW-посилання:

```text
https://raw.githubusercontent.com/USER/REPOSITORY/main/melb_data.csv
```

або якщо файл лежить у папці `data`:

```text
https://raw.githubusercontent.com/USER/REPOSITORY/main/data/melb_data.csv
```

In [ ]:
# Встав сюди своє RAW-посилання на melb_data.csv з GitHub
RAW_URL = "PASTE_YOUR_RAW_URL_HERE"

# Читаємо CSV-файл у DataFrame
# DataFrame — це таблиця в Python, схожа на Excel
df = pd.read_csv(RAW_URL)

# Перевіряємо розмір таблиці:
# перше число — кількість рядків,
# друге число — кількість стовпців
print("Розмір таблиці:", df.shape)

# Дивимось перші 5 рядків
df.head()

## Крок 2. Перший огляд даних

🎯 **Мета:** зрозуміти структуру таблиці та знайти перші проблеми.

На цьому етапі перевіряємо:

- які є стовпці;
- які типи даних;
- скільки пропусків;
- чи є підозрілі мінімальні або максимальні значення.

🔁 **Аналогія з попередніми ноутбуками:** перед побудовою моделі ми завжди спочатку дивились на `shape`, `head`, `dtypes`, пропуски та описову статистику.

💡 **Запам'ятай:** модель не можна будувати, поки ми не зрозуміли, що лежить у таблиці.

In [ ]:
# Дивимось список стовпців
print("Стовпці:")
print(df.columns.tolist())

# Дивимось типи даних
# object зазвичай означає текст
# int64 / float64 означають числові стовпці
print("\nТипи даних:")
print(df.dtypes)

# Створюємо таблицю з кількістю та відсотком пропусків
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing

In [ ]:
# Описова статистика для числових стовпців
# Тут важливо дивитися на min і max:
# наприклад, площа 0 або рік 1196 можуть бути проблемою
df.describe()

### Висновок після першого огляду

Після первинного аналізу потрібно записати, які проблеми знайдені.

Наприклад:

- у яких стовпцях багато пропусків;
- чи є технічний стовпець-індекс;
- чи дата зчиталась як текст;
- чи є нулі там, де їх не повинно бути;
- чи є підозрілий рік побудови.

✍️ **Тут допиши свій висновок після запуску коду.**

## Крок 3. Очищення даних

🎯 **Мета:** прибрати або виправити проблеми, які можуть заважати моделі.

Що робимо:

1. Створюємо копію таблиці.
2. Прибираємо технічний індекс, якщо він є.
3. Перетворюємо `Date` у справжній тип дати.
4. Замінюємо нулі у площах на `NaN`, якщо нуль означає відсутність даних.
5. Виправляємо неможливі значення `YearBuilt`.
6. Перевіряємо дублікати.

⚠️ **Важливо:** пропуски ми не заповнюємо вручну на всіх даних одразу. Це краще робити всередині Pipeline, щоб не було витоку даних.

In [ ]:
# Робимо копію, щоб не псувати оригінальний df
clean = df.copy()

# 1. Якщо є технічний стовпець індексу, видаляємо його.
# Часто він називається 'Unnamed: 0'
if "Unnamed: 0" in clean.columns:
    clean = clean.drop(columns=["Unnamed: 0"])

# 2. Перетворюємо Date на тип дати.
# dayfirst=True означає, що дата записана у форматі день/місяць/рік
clean["Date"] = pd.to_datetime(clean["Date"], dayfirst=True, errors="coerce")

# 3. Нулі у площах часто означають не реальний нуль,
# а відсутність інформації.
# Тому замінюємо 0 на np.nan.
zero_as_missing_cols = ["Landsize", "BuildingArea"]

for col in zero_as_missing_cols:
    if col in clean.columns:
        clean.loc[clean[col] == 0, col] = np.nan

# 4. Неможливі роки побудови замінюємо на пропуски.
# Наприклад, будинки з роком побудови раніше 1800
# можна вважати помилкою введення.
if "YearBuilt" in clean.columns:
    clean.loc[clean["YearBuilt"] < 1800, "YearBuilt"] = np.nan

# 5. Перевіряємо дублікати
print("Кількість дублікатів:", clean.duplicated().sum())

# Якщо дублікати є, прибираємо їх
clean = clean.drop_duplicates()

print("Розмір після очищення:", clean.shape)

clean.head()

### Висновок після очищення

Після цього кроку таблиця стала чистішою:

- дата перетворена у правильний формат;
- технічні стовпці видалені;
- нулі у площах замінені на пропуски;
- неможливі роки побудови замінені на `NaN`;
- дублікати перевірені.

✍️ **Тут допиши, які саме проблеми були знайдені у твоїх даних після запуску коду.**

## Крок 4. EDA — дослідницький аналіз даних

🎯 **Мета:** зрозуміти дані перед моделюванням.

Будуємо прості графіки та дивимось:

- розподіл ціни;
- логарифм ціни;
- залежність ціни від типу житла;
- кореляцію числових ознак із ціною;
- ознаки, які можуть дублювати одна одну.

💡 **Запам'ятай:** EDA потрібен не для моделі напряму, а для розуміння даних людиною.

In [ ]:
# 1. Розподіл ціни
plt.figure(figsize=(8, 4))
plt.hist(clean["Price"], bins=50)
plt.title("Розподіл ціни житла")
plt.xlabel("Price")
plt.ylabel("Кількість об'єктів")
plt.show()

# 2. Розподіл логарифма ціни
# Часто ціни мають довгий хвіст, тому логарифм робить розподіл рівнішим
plt.figure(figsize=(8, 4))
plt.hist(np.log10(clean["Price"]), bins=50)
plt.title("Розподіл log10(Price)")
plt.xlabel("log10(Price)")
plt.ylabel("Кількість об'єктів")
plt.show()

In [ ]:
# 3. Boxplot ціни за типом житла
# Це допомагає побачити, які типи житла дорожчі або дешевші

plt.figure(figsize=(8, 5))
clean.boxplot(column="Price", by="Type")
plt.title("Ціна залежно від типу житла")
plt.suptitle("")
plt.xlabel("Type")
plt.ylabel("Price")
plt.show()

In [ ]:
# 4. Кореляція числових ознак із ціною

num = clean.select_dtypes("number")

cor_price = (
    num.corr()["Price"]
    .drop("Price")
    .sort_values(key=abs, ascending=False)
)

print("Кореляція числових ознак із Price:")
print(cor_price.round(3))

In [ ]:
# 5. Матриця кореляцій числових ознак
# Тут шукаємо пари ознак, які майже дублюють одна одну,
# наприклад Rooms і Bedroom2

corr_matrix = num.corr().round(2)
corr_matrix

### Висновки після EDA

✍️ Після запуску графіків допиши 2–3 висновки:

Наприклад:

- розподіл `Price` скошений вправо: більшість об'єктів дешевші, але є дуже дорогі;
- тип житла впливає на ціну;
- деякі числові ознаки мають помітний зв'язок із ціною;
- `Rooms` і `Bedroom2` можуть бути схожими ознаками.

⚠️ **Важливо:** кореляція не означає причинність. Вона лише показує лінійний зв'язок.

## Крок 5. Feature Engineering — створення нових ознак

🎯 **Мета:** створити нові корисні стовпці, які можуть допомогти моделі.

Створимо:

- `sale_year` — рік продажу;
- `sale_month` — місяць продажу;
- `age` — вік будинку на момент продажу;
- `has_buildingarea` — чи була вказана площа будинку;
- `has_yearbuilt` — чи був вказаний рік побудови;
- `suburb_freq` — частота району.

⚠️ **Важливо:** не створюємо ознак із `Price`, бо це витік даних.

In [ ]:
# Створюємо копію очищених даних
fe = clean.copy()

# 1. Рік та місяць продажу
fe["sale_year"] = fe["Date"].dt.year
fe["sale_month"] = fe["Date"].dt.month

# 2. Вік будинку на момент продажу
# age = рік продажу - рік побудови
fe["age"] = fe["sale_year"] - fe["YearBuilt"]

# Якщо вік вийшов від'ємним, це помилка — замінюємо на NaN
fe.loc[fe["age"] < 0, "age"] = np.nan

# 3. Прапорці наявності даних
# 1 — значення було, 0 — значення було пропущене
fe["has_buildingarea"] = fe["BuildingArea"].notna().astype(int)
fe["has_yearbuilt"] = fe["YearBuilt"].notna().astype(int)

# 4. Частотне кодування району Suburb
# Замість сотень окремих стовпців для районів
# створюємо один числовий стовпець: як часто цей район зустрічається
fe["suburb_freq"] = fe["Suburb"].map(fe["Suburb"].value_counts())

# Перевіряємо результат
fe[["Date", "sale_year", "sale_month", "YearBuilt", "age", "has_buildingarea", "has_yearbuilt", "Suburb", "suburb_freq"]].head()

In [ ]:
# Прибираємо стовпці, які не будемо використовувати напряму в моделі.
# Address, SellerG, Suburb — текстові ознаки з великою кількістю унікальних значень.
# Date вже розкладена на sale_year і sale_month.
# Bedroom2 може дублювати Rooms.

cols_to_drop = [
    "Address",
    "SellerG",
    "Suburb",
    "Date",
    "Bedroom2"
]

# Видаляємо лише ті стовпці, які реально є в таблиці
fe = fe.drop(columns=[col for col in cols_to_drop if col in fe.columns])

print("Розмір після Feature Engineering:", fe.shape)
fe.head()

### Висновок після Feature Engineering

Ми створили нові ознаки, які можуть краще пояснювати ціну житла.

📌 Особливо важливі:

- вік будинку;
- місяць і рік продажу;
- частота району;
- факт наявності або відсутності даних про площу та рік побудови.

💡 **Запам'ятай:** хороші ознаки часто важливіші за складну модель.

## Крок 6. Підготовка до моделювання

🎯 **Мета:** підготувати `X` та `y`, розділити дані на train/test і створити Pipeline для обробки ознак.

- `y` — ціль, тобто `Price`.
- `X` — усі ознаки, крім `Price`.

Категорійні ознаки потрібно закодувати через One-Hot Encoding, а пропуски заповнити через SimpleImputer.

🔁 **Аналогія з попередніми роботами:** як і раніше, Pipeline потрібен, щоб однаково обробляти train і test та не допустити витоку даних.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 1. Визначаємо ціль і ознаки
y = fe["Price"]
X = fe.drop(columns=["Price"])

# 2. Визначаємо категорійні та числові стовпці
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Категорійні стовпці:", cat_cols)
print("Числові стовпці:", num_cols)

# 3. Pipeline для числових ознак:
# заповнюємо пропуски медіаною
num_transformer = SimpleImputer(strategy="median")

# 4. Pipeline для категорійних ознак:
# спочатку заповнюємо пропуски найчастішим значенням,
# потім робимо One-Hot Encoding
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 5. Об'єднуємо обробку числових і категорійних ознак
preprocess = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

# 6. Ділимо дані на train і test
# train — для навчання,
# test — для чесної перевірки моделі
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

### Висновок після підготовки

Дані підготовлені до моделювання:

- виділено ціль `Price`;
- ознаки розділено на числові й категорійні;
- створено `ColumnTransformer`;
- дані поділено на навчальну та тестову вибірки.

⚠️ **Важливо:** усі заповнення пропусків і кодування відбуватимуться всередині Pipeline, а не вручну на всіх даних одразу.

## Крок 7. Навчання моделей і оцінка якості

🎯 **Мета:** порівняти кілька моделей регресії.

Будемо використовувати:

1. `DummyRegressor` — базова модель, яка прогнозує середнє.
2. `LinearRegression` — проста лінійна модель.
3. `RandomForestRegressor` — складніша модель на основі дерев.

Метрики:

- **MAE** — середня абсолютна помилка.
- **RMSE** — сильніше карає великі помилки.
- **R²** — наскільки добре модель пояснює зміну ціни.

💡 **Запам'ятай:** модель має бути кращою за базову лінію.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(model, name):
    """
    Функція:
    1. Створює Pipeline з preprocessing + model.
    2. Навчає модель на train.
    3. Робить прогноз на test.
    4. Рахує MAE, RMSE, R2.
    5. Повертає навчений Pipeline.
    """

    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    # Навчаємо тільки на тренувальних даних
    pipe.fit(X_train, y_train)

    # Прогнозуємо на тестових даних
    pred = pipe.predict(X_test)

    # Рахуємо метрики
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)

    print(f"Модель: {name}")
    print(f"MAE:  {mae:,.0f}")
    print(f"RMSE: {rmse:,.0f}")
    print(f"R2:   {r2:.3f}")
    print("-" * 40)

    return pipe

# 1. Базова модель
dummy_pipe = evaluate(
    DummyRegressor(strategy="mean"),
    "Dummy Regressor"
)

# 2. Лінійна регресія
linear_pipe = evaluate(
    LinearRegression(),
    "Linear Regression"
)

# 3. Випадковий ліс
forest_pipe = evaluate(
    RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Random Forest"
)

### Висновок після оцінки моделей

Після запуску коду потрібно порівняти MAE, RMSE та R².

✍️ Допиши свій висновок:

- яка модель має найменшу MAE;
- яка модель має найменшу RMSE;
- яка модель має найбільший R²;
- чи складна модель краща за базову.

Зазвичай `Random Forest` має працювати краще за `DummyRegressor` та часто краще за просту лінійну регресію.

## Крок 8. Важливість ознак у Random Forest

🎯 **Мета:** зрозуміти, які ознаки найбільше впливають на прогноз ціни.

Random Forest має атрибут `feature_importances_`, який показує внесок кожної ознаки.

⚠️ **Важливо:** після One-Hot Encoding кількість ознак збільшується, тому потрібно дістати нові назви стовпців із Pipeline.

In [ ]:
# Дістаємо навчену модель Random Forest з Pipeline
rf_model = forest_pipe.named_steps["model"]

# Дістаємо назви ознак після preprocessing
feature_names = forest_pipe.named_steps["preprocess"].get_feature_names_out()

# Створюємо таблицю важливості ознак
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

# Дивимось топ-15 ознак
importance.head(15)

In [ ]:
# Будуємо графік топ-15 важливих ознак

top_imp = importance.head(15)

plt.figure(figsize=(8, 6))
plt.barh(top_imp["feature"], top_imp["importance"])
plt.gca().invert_yaxis()
plt.title("Top-15 важливих ознак для Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

### Висновок щодо важливості ознак

✍️ Після запуску коду допиши:

- які ознаки найважливіші;
- чи важливі площа, кількість кімнат, відстань до центру;
- чи впливають район або тип житла;
- чи логічний результат з точки зору ринку нерухомості.

## Крок 9. Прогноз для одного прикладу

🎯 **Мета:** показати, як модель прогнозує ціну для одного конкретного об'єкта.

Беремо один рядок із тестової вибірки та порівнюємо:

- реальну ціну;
- прогноз моделі;
- різницю між ними.

In [ ]:
# Беремо перший об'єкт із тестової вибірки
example = X_test.iloc[[0]]

# Реальна ціна
real_price = y_test.iloc[0]

# Прогноз Random Forest
predicted_price = forest_pipe.predict(example)[0]

print("Реальна ціна:", round(real_price, 2))
print("Прогноз моделі:", round(predicted_price, 2))
print("Помилка:", round(abs(real_price - predicted_price), 2))

example

### Висновок по прикладу

Модель зробила прогноз для одного об'єкта з тестової вибірки.

✍️ Допиши:

- наскільки прогноз відрізняється від реальної ціни;
- чи є ця помилка прийнятною;
- які характеристики об'єкта могли вплинути на прогноз.

# Загальний висновок

У цьому проєкті було виконано повний цикл роботи з даними для задачі регресії.

Ми:

1. Завантажили дані про житло в Мельбурні.
2. Провели первинний огляд таблиці.
3. Очистили дані від технічних проблем.
4. Провели EDA та побудували базові графіки.
5. Створили нові ознаки.
6. Побудували Pipeline для обробки даних.
7. Навчили кілька моделей регресії.
8. Порівняли їх за MAE, RMSE та R².
9. Проаналізували важливість ознак.
10. Зробили прогноз для одного прикладу.

📌 **Фінальний висновок після запуску всього ноутбука:**

✍️ Тут допиши, яка модель виявилася найкращою, яку помилку вона показала та які ознаки найбільше впливають на ціну житла.

💡 **Головна ідея:** у задачах регресії важливо не тільки побудувати модель, а й чесно оцінити її на тестових даних і порівняти з базовою лінією.

# Чек-лист готовності

- [ ] Дані завантажені через RAW-посилання.
- [ ] Проведено перший огляд.
- [ ] Дані очищені.
- [ ] Побудовано щонайменше 3 графіки EDA.
- [ ] Створено нові ознаки.
- [ ] Побудовано Pipeline.
- [ ] Дані поділено на train/test.
- [ ] Навчено базову модель.
- [ ] Навчено Linear Regression.
- [ ] Навчено Random Forest.
- [ ] Пораховано MAE, RMSE, R².
- [ ] Проаналізовано важливість ознак.
- [ ] Зроблено прогноз для одного прикладу.
- [ ] Написано фінальні висновки.